# CAD-ViT — riset P1–P4

**Status: NOT RUN.** Notebook riset ini mendokumentasikan kandidat Phase 1/2 di branch `preof-kc` (snapshot HEAD `3a563c6`, source Phase 1/2 masih untracked); belum ada training, kurva, confusion matrix, ataupun hasil final test P1–P4. Sel kode hanya untuk introspeksi CPU, hash baca-saja opsional, dan kapasitas sistem. Perintah GPU/training adalah petunjuk manual untuk tahap berikutnya. Baseline [CAD_ViT.ipynb](CAD_ViT.ipynb), [reproduction/cadvit](reproduction/cadvit/), checkpoint, dan hasil historis tetap terpisah.

## Navigasi

1. [Paper dan masalah riset](#1-paper-dan-masalah-riset)
2. [Arsitektur serta formulasi](#2-arsitektur-serta-formulasi)
3. [Dataset dan provenance](#3-dataset-dan-provenance)
4. [Ablasi dan evaluasi](#4-ablasi-dan-evaluasi)
5. [Kesiapan GPU serta perintah KC](#5-kesiapan-gpu-serta-perintah-kc)
6. [Hasil dan batas ilmiah](#6-hasil-dan-batas-ilmiah)

## 1. Paper dan masalah riset

Y. Wang, X. Gou, W. Li, “CAD-ViT: Coordinate Attention-Enhanced Vision Transformer With Dilated Feature Fusion for Diabetic Retinopathy Staging,” *IEEE Access* 13 (2025), 91505–91514, DOI [10.1109/ACCESS.2025.3572455](https://doi.org/10.1109/ACCESS.2025.3572455). §III/Gambar 2–3 mendeskripsikan dilated feature fusion, coordinate attention, dan Transformer; §IV membahas APTOS2019 serta DDR.

**Spesifikasi paper:** alur sembilan blok pada Gambar 3, dilatasi 3/5/7/9, coordinate attention r=16, dan lima grade. **Asumsi reproduksi:** lebar kanal, stride/expansion, patch/token layout, Transformer depth/heads/FFN, dan classifier internal; lihat [audit fidelity](docs/ARCHITECTURE_FIDELITY_AUDIT.md). Tabel 1 paper: sekitar **7,2629 juta** parameter, sedangkan `CADViTCandidate`: **5.282.101**. V2 BG10 adalah ekstensi preprocessing, bukan langkah yang dapat diatribusikan ke paper. Penelitian ini menguji kandidat reproduksi parsial, bukan source resmi penulis.

**Pertanyaan eksploratori:** dapatkah perubahan terkontrol meningkatkan validation macro-F1 tanpa memilih metode pada test set?

| Komponen | Hipotesis yang akan diuji | Kontrol |
| --- | --- | --- |
| P1 local contrast | Jalur high-pass mungkin membawa detail yang berguna; belum ada bukti detail itu lesi. | Baseline vs P1 |
| P2 adaptive MFB | Bobot cabang per sampel mungkin lebih berguna daripada cabang tetap. | Baseline vs P2 |
| P3 boundary-conditioned modulation | Prediksi batas grade lokal mungkin menambah manfaat setelah supervisi ordinal dan gating biasa dikendalikan. | Probe vs full, dengan/tanpa P2 |
| P4 ordinal head | Auxiliary objective global mungkin membantu klasifikasi utama. | Baseline vs P4 |

Hipotesis nol setiap kontras: tidak ada peningkatan pada metrik validasi yang dipraspesifikasi. Semua hasil validation kelak bersifat eksploratori karena split itu juga memilih checkpoint awal.

## 2. Arsitektur serta formulasi

Diagram statis ini mengikuti `CADViTCandidate.block_names` dan letak modul di [novelty/cadvit_model.py](novelty/cadvit_model.py). Sel selanjutnya dapat membuat ulang diagram dari objek model aktual. Panah P1 berarti **penjumlahan setelah stem**, bukan penggantian input stem.

```mermaid
flowchart LR
  I[RGB 512×512] --> S[Stem] --> A[IRB-1] --> B[IRB+A-1 / P3]
  I -. P1 high-pass residual .-> S
  B --> C[IRB-2] --> D[IRB+A-2 / P3] --> E[EMB-1 / MFB P2]
  E --> F[IRB+A-3 / P3] --> G[EMB-2 / MFB P2]
  G --> H[IRB+A-4 / P3] --> J[EMB-3 / MFB P2]
  J --> K[Head 320] --> L[Average pool] --> M[5 logits]
  L -. P4 auxiliary .-> N[4 ordinal logits]
```

Empat P3 probe muncul pada IRB+A; tiga P2 gate pada MFB di EMB. P4 tidak mengganti classifier lima kelas.

### 2.1 Source impor dan grafik programatik

Sel ini menemukan root dan mengimpor implementasi yang sudah ada. Sel berikutnya menginstansiasi model **di CPU, tanpa checkpoint atau forward**, mengambil nama blok, dan memeriksa jumlah MFB/CA; tidak membaca citra atau final test.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "novelty/cadvit_model.py").is_file()
             and (p / "reproduction/cadvit/model.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside the novelty repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import Markdown, display
from reproduction.cadvit.model import (
    CADViTCandidate, CoordinateAttention, EnhancedMobileViTBlock, InvertedResidualBlock)
from novelty.cadvit_model import NovelCADViT, PRESETS, parameter_counts
from novelty.cadvit_pilot import CHECKPOINTS, PRIMARY_ABLATIONS
print("Root:", ROOT, "| pilot arms:", ", ".join(PRIMARY_ABLATIONS))

**Tujuan sel diagram.** Struktur baseline dibaca dari `block_names` dan tipe blok aktual; tiga lokasi MFB dan empat lokasi Coordinate Attention diverifikasi sebelum Mermaid ditampilkan. Ini pemeriksaan arsitektur statis, bukan forward pass atau bukti performa.

In [ ]:
base = CADViTCandidate()
blocks = list(zip(base.block_names, base.blocks))
mfb = [n for n, b in blocks if isinstance(b, EnhancedMobileViTBlock)]
ca = [n for n, b in blocks if isinstance(b, InvertedResidualBlock)
      and isinstance(b.attention, CoordinateAttention)]
assert len(blocks) == 9 and len(mfb) == 3 and len(ca) == 4
labels = ["Stem"] + [n + (" / P2" if n in mfb else " / P3" if n in ca else "")
                     for n, _ in blocks] + ["Head", "Pool", "5 logits"]
arrows = "\n".join(f'  n{i}["{v}"] --> n{i+1}["{labels[i+1]}"]'
                   for i, v in enumerate(labels[:-1]))
display(Markdown("Grafik dari nama blok aktual:\n\n```mermaid\nflowchart LR\n"
                 + arrows + "\n```"))
print("MFB:", mfb, "| CA:", ca)
del base

### 2.2 Formulasi dan hubungan P1–P4

**P1:** `D=I−AvgPool₃×₃(I)`; `S′(I)=S(I)+Conv₃×₃,stride₂(D)`. Proyeksi diawali nol sehingga output awal stem sama. Jalur ini mengekspos kontras lokal; keberhasilan mempertahankan lesi belum diketahui.

**P2:** untuk empat cabang asli `Fᵢ(X)` (dilatasi 3/5/7/9), `αᵢ=4·softmax(Linear(GAP(X)))ᵢ`; `MFB′=Fuse(Concat[αᵢFᵢ(X)])`. Gate nol pada awal memberi semua `αᵢ=1`, menjaga urutan serta bobot cabang/fusion. Ini kontrol ordinary adaptive gating.

**P3:** pada fitur `U` **sebelum** coordinate attention, `s=Linear(GAP(U))`, threshold berurutan `τ₀=a`, `τₖ=a+Σⱼ₌₁ᵏsoftplus(dⱼ)`, logit `zₖ=s−τₖ`, target train-only `tₖ=1[y>k]`. Empat probe menerima BCE. Probe-only mengeluarkan CA asli; full mengeluarkan `CA(U)·(1+tanh(Linear(sigmoid(z))))` per kanal. Modulasi diawali nol. Ini modulasi **keluaran fitur CA**, bukan perubahan gate spasial CA.

**P4:** head `Linear(h)` dari pooled feature `h∈R³²⁰` menghasilkan empat logit ordinal terhadap `tₖ` yang sama. Head lima kelas tetap primer.

**Loss eksploratori:** `L=CE(logit₅,y)+0,20·mean₄site BCE(z,t)[P3]+0,20·BCE(o,t)[P4]`. P3/P4 memakai target identik di kedalaman berbeda, sehingga kombinasi dapat menggandakan tekanan supervisi. P1/P2/P3/P4 bersama tidak mengisolasi kontribusi P3. Inisialisasi output identik dengan baseline tidak menjamin lintasan BatchNorm/optimisasi tetap identik.

### 2.3 Jumlah parameter

| Arm | Total parameter |
| --- | ---: |
| Baseline | 5.282.101 |
| P1 | 5.282.965 |
| P2 | 5.283.649 |
| P3 probe-only | 5.282.761 |
| P3 full | 5.285.961 |
| P4 | 5.283.385 |
| P2+P3 probe | 5.284.309 |
| P2+P3 full | 5.287.509 |
| P1+P2+P3+P4 | 5.289.657 |
| Paper Tabel 1, r=16 | ~7.262.900, angka dibulatkan |

Sel ini menghitung ulang tiap arm dari kelas yang diimpor. Tidak ada forward, GPU, bobot checkpoint, atau data.

In [ ]:
expected = dict(baseline=5282101, p1_only=5282965, p2_only=5283649,
                p3_probe_only=5282761, p3_only=5285961, p4_only=5283385,
                p2_p3_probe=5284309, p2_p3_full=5287509, p1234=5289657)
for arm in PRIMARY_ABLATIONS:
    model = CADViTCandidate() if arm == "baseline" else NovelCADViT(PRESETS[arm])
    counts = parameter_counts(model)
    assert counts["total"] == expected[arm], (arm, counts)
    print(f"{arm:16s}: total {counts['total']:,}; trainable {counts['trainable']:,}")
    del model

## 3. Dataset dan provenance

APTOS2019: **2.930 train / 366 validation**. DDR: **10.018 train / 1.252 validation**. Manifest seed 42 dibekukan di `~/Kel_13/cad_vit_runs/manifests/<dataset>_seed42.csv`. `load_train_val_manifest` memverifikasi manifest+sidecar, hash semua citra train/validation, serta pemisahan group/konten; baris **test dilewati sebelum path/label digunakan**. Tidak ada split baru, oversampling, atau weighted CE pada pilot.

Runner memanggil `preprocess_fundus_v2` dengan `PREPROCESS_CONFIG` dan `BG_V2_CONFIG`; train lalu mendapat horizontal/vertical flip p=0,5 dan affine degrees=15/shear=10 bilinear fill=0, diikuti RGB CHW float32/255. Validation deterministik tanpa augmentasi. Transformasi geometri sesuai runner CAD-ViT yang diekstrak; kesetaraan tepat dengan implementasi historis V2 belum terbukti.

| Dataset | SHA-256 manifest CSV | Checkpoint arsip V2 BG10 dan SHA-256 | Epoch zero-based |
| --- | --- | --- | ---: |
| APTOS2019 | `73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854` | `experiment_archive_80ep_20261105/APTOS/best.pt` · `e72573f0f1c26baac53e18d950076845952da503261e212021ff6320c9ae72d5` | 68 |
| DDR | `7bc3e3916e012bc46fcd9cd69aa469e0daf3229e1273f1d8ecb6054899bf6eb1` | `experiment_archive_80ep_20261105/DDR/best.pt` · `844665f0c597142915286b727a21f1d2223f43e0f057d083b6c1dd035c180ac1` | 71 |

Checkpoint hash, model mapping, V2 processing, manifest hash, dataset, seed, LR, batch, dan metadata recovery DDR menjadi guard runner. DDR checkpoint memiliki riwayat mixed weighted-FP32 recovery; pilot baru bukan kelanjutan optimizer tersebut. Hash membuktikan identitas byte dengan nilai rujukan, tidak sendiri membuktikan seluruh lineage.

### 3.1 Hash check baca-saja, opt-in

Default sel ini **SKIPPED**. Bila diperlukan, aktifkan flag dan `CADVIT_ARTIFACT_ROOT` untuk menghash dua manifest CSV train/validation dan dua checkpoint di atas; tidak membaca citra/prediksi test. Preflight CLI menjalankan validasi train/validation yang lebih lengkap.

In [ ]:
VERIFY_LOCAL_PROVENANCE = False
if VERIFY_LOCAL_PROVENANCE:
    import os
    from novelty.cadvit_training import file_sha256
    from novelty.preof_phase1 import SPECS
    ARTIFACT_ROOT = Path(os.environ["CADVIT_ARTIFACT_ROOT"]).expanduser().resolve()
    for dataset in ("APTOS2019", "DDR"):
        sources = (
            (ARTIFACT_ROOT / "cad_vit_runs" / "manifests" / f"{dataset}_seed42.csv",
             SPECS[dataset]["manifest_sha256"]),
            (ARTIFACT_ROOT / CHECKPOINTS[dataset]["relative"], CHECKPOINTS[dataset]["sha256"]),
        )
        for path, expected_sha in sources:
            observed = file_sha256(path)
            assert observed == expected_sha, (dataset, path, observed)
            print(dataset, path.name, "SHA-256 PASS", observed)
else:
    print("SKIPPED: set VERIFY_LOCAL_PROVENANCE=True and CADVIT_ARTIFACT_ROOT.")

## 4. Ablasi dan evaluasi

| Arm CLI | P1 | P2 | P3 probe BCE | P3 modulation | P4 BCE | Tujuan |
| --- | :---: | :---: | :---: | :---: | :---: | --- |
| `baseline` | — | — | — | — | — | Kontrol asli |
| `p1_only` | ✓ | — | — | — | — | Detail |
| `p2_only` | — | ✓ | — | — | — | Gating biasa |
| `p3_probe_only` | — | — | ✓ | — | — | Supervisi lokal |
| `p3_only` | — | — | ✓ | ✓ | — | Modulasi di atas probe |
| `p4_only` | — | — | — | — | ✓ | Supervisi ordinal global |
| `p2_p3_probe` | — | ✓ | ✓ | — | — | Gating + probe |
| `p2_p3_full` | — | ✓ | ✓ | ✓ | — | Modulasi di atas gating + probe |
| `p1234` | ✓ | ✓ | ✓ | ✓ | ✓ | Kombinasi eksploratori |

Kontras utama: `p2_p3_full` vs `p2_p3_probe`; kontras kedua `p3_only` vs `p3_probe_only`. Batas lain harus sama dalam tiap dataset: checkpoint, manifest, seed, microbatch, augmentasi, optimizer, epoch, aturan seleksi. Jangan memilih arm dengan final test.

Pilot: seed 42, AdamW LR 0,000186, weight decay 0, FP32, grad clip 1, effective batch 16, microbatch pembagi 16, **1–5 epoch**. Per epoch tersimpan loss, accuracy, macro/weighted F1, metrik per kelas (precision/recall/F1/specificity/support), confusion matrix true-row/predicted-column, QWK, grade MAE, waktu, dan peak allocated VRAM. `best.pt`: validation macro-F1 tertinggi; tie persis: primary validation CE loss lebih rendah, lalu epoch lebih awal. `last.pt`, config, JSONL history, summary disimpan luar repo. Resume menolak config/history/best/last atau RNG state yang tidak cocok.

## 5. Kesiapan GPU serta perintah KC

Snapshot infrastruktur: kedua RTX 3060 bersama sedang dipakai; `/mnt/home` ~99% terpakai, ~22 GiB bebas. Ini bukan angka tetap. **Synthetic 512×512 GPU forward/backward batch 1 masih PENDING**; CPU 512×512 dan GPU 64×64 tidak membuktikan kebutuhan memori GPU penuh. Jangan mengganggu proses pengguna lain. Tidak ada ambang VRAM aman yang telah diukur; cek ketersediaan aktual serta disk sebelum smoke/pilot.

`CUDA_VISIBLE_DEVICES=<approved_index>` memilih GPU fisik; di dalam proses ia menjadi `cuda:0`. `--device cuda` tidak fallback ke CPU. `preflight` memeriksa hash train/validation dan checkpoint tanpa inference; `smoke` hanya forward/backward **tensor sintetis 512×512**, tanpa optimizer step/data; `pilot`/`resume` adalah training manual yang belum diizinkan pada tahap ini.

### 5.1 Snapshot kapasitas baca-saja

Sel ini hanya membaca disk dan `nvidia-smi`; tidak memulai workload GPU. Tidak memakai batas siap-training yang belum terukur.

In [ ]:
import os, shutil, subprocess
ARTIFACT_ROOT = Path(os.environ.get("CADVIT_ARTIFACT_ROOT",
                                    "/mnt/home/probation-03/Kel_13")).expanduser().resolve()
usage = shutil.disk_usage(ARTIFACT_ROOT)
print(f"Filesystem free: {usage.free / 2**30:.2f} GiB ({usage.free / usage.total:.1%})")
probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,memory.free,utilization.gpu",
     "--format=csv,noheader,nounits"], capture_output=True, text=True, check=False)
print(probe.stdout.strip() if probe.returncode == 0 else "nvidia-smi unavailable")
print("READ-ONLY snapshot; 512×512 GPU smoke is still required.")

### 5.2 Perintah manual APTOS2019

Jalankan dari root worktree KC; pilih `GPU` hanya setelah memeriksa pengguna lain. **Jangan jalankan pilot/resume sekarang.** Untuk arm lain ubah `--ablation`; runner memberi direktori independen.

```bash
cd /mnt/home/probation-03/Kel_13/Diabetic-Rethinography-Novelty
ROOT=/mnt/home/probation-03/Kel_13
OUT="$ROOT/cadvit_novelty_pilots"
read -r -p "Approved free GPU index (0 or 1): " GPU
../.venv/bin/python -m novelty.cadvit_pilot preflight --artifact-root "$ROOT" --output-root "$OUT" --dataset APTOS2019 --ablation p2_p3_probe --micro-batch 1 --epochs 2
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot smoke --artifact-root "$ROOT" --output-root "$OUT" --dataset APTOS2019 --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
# Hanya setelah gate 512×512 GPU dan review ilmiah:
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot pilot --artifact-root "$ROOT" --output-root "$OUT" --dataset APTOS2019 --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot resume --artifact-root "$ROOT" --output-root "$OUT" --dataset APTOS2019 --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
```

### 5.3 Perintah manual DDR

DDR memakai manifest/checkpoint berbeda dan output subdirektori sendiri. Recovery checkpoint DDR weighted FP32; pilot baru CE biasa. **Jangan jalankan pilot/resume sekarang.**

```bash
cd /mnt/home/probation-03/Kel_13/Diabetic-Rethinography-Novelty
ROOT=/mnt/home/probation-03/Kel_13
OUT="$ROOT/cadvit_novelty_pilots"
read -r -p "Approved free GPU index (0 or 1): " GPU
../.venv/bin/python -m novelty.cadvit_pilot preflight --artifact-root "$ROOT" --output-root "$OUT" --dataset DDR --ablation p2_p3_probe --micro-batch 1 --epochs 2
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot smoke --artifact-root "$ROOT" --output-root "$OUT" --dataset DDR --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
# Hanya setelah gate 512×512 GPU dan review ilmiah:
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot pilot --artifact-root "$ROOT" --output-root "$OUT" --dataset DDR --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot resume --artifact-root "$ROOT" --output-root "$OUT" --dataset DDR --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
```

Output: `$OUT/<dataset>/p2_p3_probe_seed42_e2_mb1_eff16_fp32/`. `pilot` menolak direktori yang ada; `resume` memerlukan config, history, best/last, source dan RNG yang kompatibel. Mengubah epoch/microbatch menghasilkan konfigurasi/direktori berbeda. Output harus di luar repository/dataset/arsip/foundation. Estimasi dua checkpoint FP32+AdamW: ~125–150 MiB per arm, bukan ukuran terukur.

## 6. Hasil dan batas ilmiah

| Dataset | Arm | Training | Validation | Plot / confusion matrix |
| --- | --- | --- | --- | --- |
| APTOS2019 | Semua P1–P4 | **NOT RUN** | **NOT RUN** | **NOT AVAILABLE** |
| DDR | Semua P1–P4 | **NOT RUN** | **NOT RUN** | **NOT AVAILABLE** |

Tidak ada hasil atau grafik P1–P4 yang sah untuk diinterpretasikan. Hasil reproduksi historis tidak boleh dipindahkan ke tabel ini.

**Prior art:** P2 terkait [adaptive multiscale weighting DR](https://www.mdpi.com/2079-9292/10/12/1369); [coordinate attention](https://openaccess.thecvf.com/content/CVPR2021/papers/Hou_Coordinate_Attention_for_Efficient_Mobile_Network_Design_CVPR_2021_paper.pdf) dan [target ordinal kumulatif](https://openaccess.thecvf.com/content_ECCVW_2018/papers/11134/Liu_Ordinal_Regression_with_Neuron_Stick-breaking_for_Medical_Diagnosis_ECCVW_2018_paper.pdf) juga sudah dikenal. Mekanisme yang diuji adalah manfaat tambahan **modulasi fitur berbasis prediksi batas grade lokal** setelah ordinary gating dan supervisi ordinal dikendalikan. Audit prior art lebih luas serta konfirmasi independen tetap diperlukan sebelum klaim kontribusi metodologis.

**Keterbatasan:** P1 mungkin menangkap noise; P2 mungkin hanya mereweight cabang lama; P3 probe dapat kurang terkalibrasi; P4 bisa bersaing dengan CE; P3/P4 mengulang target ordinal; BatchNorm bisa mengubah lintasan training walau output awal setara. Selisih parameter 5,28M vs ~7,26M, preprocessing BG10, recovery DDR campuran, resep augmentasi V2 yang belum diverifikasi, dan ketidakpastian split paper membatasi kesetaraan reproduksi. Validation yang juga memilih checkpoint awal bukan konfirmasi independen. Tidak ada bukti pelestarian lesi, kausalitas klinis, atau kesiapan klinis.

**Reproduksibilitas:** sebelum pilot, ulangi preflight APTOS/DDR, catat hash source/manifest/checkpoint, image count, library versions, free disk, dan peak VRAM smoke sintetis 512×512. Bekukan arm/microbatch/epoch sebelum training. Isi bagian hasil hanya dari artefak run yang benar-benar ada. Jangan memakai final test untuk pengembangan. Source: [model](novelty/cadvit_model.py), [loss/transfer](novelty/cadvit_training.py), [runner](novelty/cadvit_pilot.py), [tes model](tests/test_cadvit_novelty.py), [tes runner](tests/test_cadvit_pilot.py), [protokol](docs/CADVIT_NOVELTY_PROTOCOL.md).